# 21 — Throughput (records / s): jev-lite vs NLI vs Laya (local, Apple Silicon) vs JEV (API)

How many records per second each model gets through when it may work on many at once. The companion of
`20_speed_benchmark.ipynb`, which measures latency of one record at a time.

- **Local models (MPS):** real batches of *different* records (different states), sorted by length so padding stays small,
  at batch sizes 1, 4, 8, 16, 32, 64. Each primitive is timed on its own, then summed: overall records/s =
  all records / total time. Includes tokenisation and the read-back of probabilities. NLI's batch is in
  premise–hypothesis *pairs* (one per option), as its model sees them.
- **JEV (API):** the same records with 1, 4, 8, 16, 32 requests in flight. **This sends real API calls:**
  `N_JEV_PER_PRIMITIVE × 3 × len(JEV_CONCURRENCY)` (default 50 × 3 × 5 = 750 calls, ~0.35M input tokens).
  Off by default: set `RUN_JEV = True`.

Records: the same 300 as in notebook 20 (100 per primitive, seed 0). Results go to `data/speed/apple_silicon/`.
Before running: plug in the power and close heavy apps.

## 1. Environment

In [1]:
STACK = "apple_silicon"

import os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")  # all three local models are in the Hugging Face cache

from jevlite.config import apply_env, load_config
from jevlite.log import logger, setup_logging

setup_logging()
cfg = load_config(STACK)
apply_env(cfg)
logger.info("stack {}: config loaded, root {}", STACK, ROOT)

11:38:27 | INFO    | stack apple_silicon: config loaded, root /Users/nefelim584/PycharmProjects/JEV_Judge


In [2]:
import platform
import subprocess

import torch


def _sysctl(key: str) -> str:
    return subprocess.run(["sysctl", "-n", key], capture_output=True, text=True).stdout.strip()


ENV = {
    "chip": _sysctl("machdep.cpu.brand_string"),
    "memory_gb": round(int(_sysctl("hw.memsize") or 0) / 2**30),
    "macos": platform.mac_ver()[0],
    "python": platform.python_version(),
    "torch": torch.__version__,
    "mps": torch.backends.mps.is_available(),
}
assert ENV["mps"], "MPS is not available: this notebook is for Apple Silicon"
ENV

{'chip': 'Apple M1 Pro',
 'memory_gb': 16,
 'macos': '26.5.2',
 'python': '3.14.0',
 'torch': '2.14.1',
 'mps': True}

## 2. Parameters

In [3]:
SAMPLE = "data/jev_api/sample.jsonl"
JEVLITE_MODEL = "data/runs/stage_a-full-s42/final"

LOCAL_MODELS = ["jevlite", "laya", "nli"]
BATCH_SIZES = [1, 4, 8, 16, 32, 64]  # records per forward pass (NLI: premise-hypothesis pairs)
N_PER_PRIMITIVE = 100                # the same 300 records as notebook 20
SEED = 0

RUN_JEV = True                      # True sends N_JEV_PER_PRIMITIVE * 3 * len(JEV_CONCURRENCY) API calls
JEV_CONCURRENCY = [1, 4, 8, 16, 32]
N_JEV_PER_PRIMITIVE = 50
JEV_MODEL = "jev-latest"

OUT = Path("data/speed/apple_silicon")
OUT.mkdir(parents=True, exist_ok=True)

## 3. Records

In [4]:
import random

import pandas as pd

from jevlite.data.unified import load_records
from jevlite.encoding import serialize_state

records = list(load_records(SAMPLE))
rng = random.Random(SEED)
bench = []
for prim in ("bool", "choice", "score"):
    pool = sorted((r for r in records if r.type == prim), key=lambda r: r.id)
    bench += rng.sample(pool, min(N_PER_PRIMITIVE, len(pool)))


def n_chars(r) -> int:
    return len(serialize_state(r.state)) + len(r.prompt) + sum(len(c) for c in r.candidates or [])


by_prim = {p: sorted((r for r in bench if r.type == p), key=n_chars) for p in ("bool", "choice", "score")}
{p: len(v) for p, v in by_prim.items()}

{'bool': 100, 'choice': 100, 'score': 100}

## 4. Batched inference, one function per local model

Each takes a list of records and returns one probability array per record. Records come in sorted by length,
so a batch holds records of similar length.

In [5]:
import numpy as np

from jevlite.baselines import load_baseline
from jevlite.inference import _normalise


def jevlite_batch(model, recs):
    rows, _ = model.predictor._probs([(r.state, r.to_question(f"q{j}")) for j, r in enumerate(recs)])
    return [np.asarray(p if r.type == "bool" else _normalise(p)) for r, p in zip(recs, rows)]


def laya_batch(model, recs):
    # LayaZeroShot.predict for several states in one forward pass (its collate_items flattens the groups)
    from jevlite.baselines.laya import to_laya_question

    rl, items, qs = model.rl, [], []
    for r in recs:
        q = to_laya_question(r.to_question("q"))
        ids, markers = rl.build_sequence(model.tokenizer, r.state, q, model.cfg["max_len"], model.cfg["head_max_len"])
        if len(markers) != len(rl.render_options(q)):
            raise ValueError("options do not fit in Laya's header")
        items.append({"ids": ids, "markers": markers, "qtype": rl.QTYPES[q["t"]], "target": [0.0] * len(markers),
                      "label": -1, "episode": 0, "ep_step": 0, "ep_len": 1, "src": "jevlite"})
        qs.append(q)
    b = rl.collate_items([items], model.tokenizer.pad_token_id)
    with torch.no_grad():
        logits, _ = model.model(b["input_ids"].to(model.device), b["attention_mask"].to(model.device),
                                b["marker_pos"].to(model.device), b["marker_mask"].to(model.device), b["qtype"].to(model.device))
    logits = logits.float().cpu().numpy()
    out = []
    for i, (item, q) in enumerate(zip(items, qs)):
        k = len(item["markers"])
        z = logits[i, :k] / model._temperature(item["qtype"], k)
        p = np.exp(z - z.max())
        p /= p.sum()
        out.append(np.array([p[1]]) if q["t"] == "noul" else p)
    return out


def nli_batch(model, recs):
    return [res[0] for res in model.predict_many([(r.state, [r.to_question("q")]) for r in recs])]


BATCH_FN = {"jevlite": jevlite_batch, "laya": laya_batch, "nli": nli_batch}

/Users/nefelim584/PycharmProjects/JEV_Judge/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Local throughput

Per model and batch size: one untimed warm-up batch per primitive (MPS compiles kernels for new shapes),
then every record of each primitive once. A batch that fails (e.g. out of memory) stops the larger sizes
for that model; a record Laya cannot handle is counted as skipped.

In [6]:
import gc
import time


def free_mps():
    gc.collect()
    torch.mps.empty_cache()


def run_local(name: str) -> list[dict]:
    free_mps()
    model = load_baseline(name, JEVLITE_MODEL if name == "jevlite" else None, "mps")
    fn = BATCH_FN[name]
    rows = []
    for bs in BATCH_SIZES:
        if name == "nli":
            model.batch_size = bs  # pairs per forward pass
        step = 64 if name == "nli" else bs  # NLI: records per predict_many call; the pairs inside are batched by bs
        try:
            for prim, recs in by_prim.items():
                fn(model, recs[:step])  # warm-up
                torch.mps.synchronize()
                done = skipped = 0
                t = time.perf_counter()
                for s in range(0, len(recs), step):
                    chunk = recs[s : s + step]
                    try:
                        fn(model, chunk)
                        done += len(chunk)
                    except ValueError:  # Laya: a record that does not fit; retry one by one
                        for r in chunk:
                            try:
                                fn(model, [r])
                                done += 1
                            except ValueError:
                                skipped += 1
                torch.mps.synchronize()
                secs = time.perf_counter() - t
                rows.append({"model": name, "parallel": bs, "primitive": prim, "records": done, "skipped": skipped,
                             "seconds": secs, "rec_s": done / secs})
            total = [r for r in rows if r["model"] == name and r["parallel"] == bs]
            logger.info("{} batch {}: {:.1f} records/s", name, bs,
                        sum(r["records"] for r in total) / sum(r["seconds"] for r in total))
        except RuntimeError as e:  # MPS out of memory and the like
            logger.warning("{} batch {}: {}; skipping larger batches", name, bs, str(e)[:200])
            break
        finally:
            free_mps()
    del model
    free_mps()
    return rows


throughput = []
for name in LOCAL_MODELS:
    throughput += run_local(name)
pd.DataFrame(throughput).pivot_table(index=["model", "parallel"], columns="primitive", values="rec_s").round(1)

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 6155.63it/s]


11:39:01 | INFO    | jevlite batch 1: 11.2 records/s
11:39:22 | INFO    | jevlite batch 4: 14.7 records/s
11:39:43 | INFO    | jevlite batch 8: 14.5 records/s
11:40:07 | INFO    | jevlite batch 16: 13.9 records/s
11:40:35 | INFO    | jevlite batch 32: 12.3 records/s
11:41:16 | INFO    | jevlite batch 64: 10.9 records/s
11:42:15 | INFO    | laya batch 1: 11.1 records/s
11:42:38 | INFO    | laya batch 4: 13.7 records/s
11:43:02 | INFO    | laya batch 8: 13.2 records/s
11:43:27 | INFO    | laya batch 16: 12.6 records/s
11:43:58 | INFO    | laya batch 32: 11.3 records/s
11:44:44 | INFO    | laya batch 64: 9.8 records/s


/Users/nefelim584/PycharmProjects/JEV_Judge/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 394/394 [00:00<00:00, 765.35it/s]


11:50:46 | INFO    | nli batch 1: 1.2 records/s
11:55:43 | INFO    | nli batch 4: 1.4 records/s


KeyboardInterrupt: 

## 6. JEV throughput (API)

`N_JEV_PER_PRIMITIVE` records per primitive (the shortest-first order of section 3 is shuffled back, so each
level sees the same mix), sent with `c` requests in flight, for each `c` in `JEV_CONCURRENCY`.
Errors (e.g. rate limits) are counted, not retried beyond the SDK's own retries.

In [ ]:
if RUN_JEV:
    import asyncio
    import sys

    from dotenv import load_dotenv
    from typesafe_sdk import AsyncTypeSafeClient

    sys.path.insert(0, str(ROOT / "scripts"))
    import predict_jev_api as jev_api

    load_dotenv()
    jev_recs = {p: random.Random(SEED).sample(v, min(N_JEV_PER_PRIMITIVE, len(v))) for p, v in by_prim.items()}

    async def jev_run(client, recs, c):
        sem = asyncio.Semaphore(c)
        errors = 0

        async def one(r):
            nonlocal errors
            async with sem:
                try:
                    await client.system_one(state=r.state, questions={"q": jev_api._question(r)})
                except Exception:
                    errors += 1

        t = time.perf_counter()
        await asyncio.gather(*(one(r) for r in recs))
        return time.perf_counter() - t, errors

    async with AsyncTypeSafeClient(api_key=os.environ["TYPESAFE_API_KEY"], model=JEV_MODEL) as client:
        await jev_run(client, jev_recs["bool"][:3], 3)  # warm the connection
        for c in JEV_CONCURRENCY:
            for prim, recs in jev_recs.items():
                secs, errors = await jev_run(client, recs, c)
                done = len(recs) - errors
                throughput.append({"model": "jev", "parallel": c, "primitive": prim, "records": done, "skipped": errors,
                                   "seconds": secs, "rec_s": done / secs})
            logger.info("jev, {} in flight: {:.1f} records/s", c,
                        sum(r["records"] for r in throughput if r["model"] == "jev" and r["parallel"] == c)
                        / sum(r["seconds"] for r in throughput if r["model"] == "jev" and r["parallel"] == c))
else:
    logger.info("RUN_JEV is False: no API calls, JEV is left out of the throughput comparison")

## 7. Results

In [ ]:
MODELS = ["nli", "laya", "jevlite", "jev"]  # colour slots as in STAGE_A.md / STAGE_A_API.md
LABELS = {"nli": "NLI (DeBERTa-v3-large)", "laya": "Laya", "jevlite": "jev-lite", "jev": "JEV (API)"}

tp = pd.DataFrame(throughput)
overall = (tp.groupby(["model", "parallel"])[["records", "seconds"]].sum()
             .assign(rec_s=lambda d: d["records"] / d["seconds"])["rec_s"].unstack("parallel"))
overall = overall.reindex([m for m in MODELS if m in overall.index]).rename(index=LABELS)
overall.round(1)

In [ ]:
best = tp.groupby(["model", "parallel"])[["records", "seconds"]].sum().assign(rec_s=lambda d: d["records"] / d["seconds"])
best_par = best["rec_s"].groupby(level="model").idxmax().map(lambda t: t[1])
per_prim = tp.assign(best=tp["model"].map(best_par)).query("parallel == best").pivot_table(
    index="model", columns="primitive", values="rec_s")
summary = pd.DataFrame({
    "best parallelism": best_par,
    "records/s at best": best["rec_s"].groupby(level="model").max(),
    "records/s at 1": best["rec_s"].xs(1, level="parallel"),
}).join(per_prim.add_prefix("records/s ")).reindex([m for m in MODELS if m in best_par.index]).rename(index=LABELS)
summary.round(1)

In [ ]:
import json

tp.to_csv(OUT / "throughput.csv", index=False)
(OUT / "throughput.json").write_text(json.dumps({
    "env": ENV, "n_records": len(bench), "batch_sizes": BATCH_SIZES,
    "jev": {"run": RUN_JEV, "concurrency": JEV_CONCURRENCY, "n_per_primitive": N_JEV_PER_PRIMITIVE},
    "summary": summary.round(2).reset_index().to_dict("records"),
}, indent=1, default=str))
print(f"→ {OUT}/throughput.csv, {OUT}/throughput.json")

## 8. Charts

1. Records per second vs parallelism: batch size for the local models, requests in flight for JEV.
2. Best records per second per model and primitive.

In [ ]:
from matplotlib.ticker import FixedLocator, NullLocator, StrMethodFormatter

from jevlite.reports.style import SURFACE, TEXT_SECONDARY, save_figure, series_colors, setup_matplotlib

plt = setup_matplotlib()
COLORS = series_colors(MODELS)
present = [m for m in MODELS if m in set(tp["model"])]

fig, ax = plt.subplots(figsize=(7.6, 4.4))
for m in present:
    d = best.xs(m, level="model")["rec_s"].sort_index()
    ax.plot(d.index, d.values, color=COLORS[m], marker="o", markersize=8, markeredgecolor=SURFACE, markeredgewidth=2,
            label=LABELS[m])
ax.set_xscale("log", base=2)
ax.set_yscale("log")
xs = sorted(set(tp["parallel"]))
ax.xaxis.set_major_locator(FixedLocator(xs))
ax.xaxis.set_minor_locator(NullLocator())
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:.0f}"))
ys = [y for y in [0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500] if best["rec_s"].min() / 1.5 <= y <= best["rec_s"].max() * 1.5]
ax.yaxis.set_major_locator(FixedLocator(ys))
ax.yaxis.set_minor_locator(NullLocator())
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
ax.set_xlabel("parallelism: batch size (local, MPS) / requests in flight (JEV)")
ax.set_ylabel("records per second")
ax.set_title(f"Throughput on {ENV['chip']} vs JEV over the API", loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=len(present))
print(save_figure(fig, OUT, "throughput_vs_parallelism")[0])
fig

In [ ]:
prims = ["bool", "choice", "score"]
pp = tp.assign(best=tp["model"].map(best_par)).query("parallel == best")
fig, ax = plt.subplots(figsize=(7.6, 0.5 * len(present) * len(prims) + 1.4))
labels, y = [], 0
for m in present:
    for p in prims:
        v = pp[(pp["model"] == m) & (pp["primitive"] == p)]["rec_s"]
        if v.empty:
            continue
        ax.barh(y, v.iloc[0], height=0.6, color=COLORS[m], edgecolor=SURFACE, linewidth=2)
        ax.text(v.iloc[0], y, f"  {v.iloc[0]:.1f}", va="center", fontsize=9, color=TEXT_SECONDARY)
        labels.append(f"{LABELS[m]} · {p}  (×{best_par[m]})")
        y += 1
ax.set_yticks(range(len(labels)), labels)
ax.set_ylim(len(labels) - 0.5, -0.5)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("records per second at each model's best parallelism (×N)")
ax.set_title("Best throughput per primitive", loc="left")
print(save_figure(fig, OUT, "throughput_by_primitive")[0])
fig

## 9. Reading the numbers

- **Local batches have a ceiling** set by this GPU: past some batch size records/s stops growing (or MPS runs out of memory).
  A CUDA GPU moves the ceiling up; the ranking of the local models should hold.
- **JEV's throughput grows with requests in flight** until the API's rate limit; it does not depend on this machine.
  Each record is also a paid call — see `STAGE_A_API.md` for tokens per record.
- **NLI's cost grows with the number of options** (one pair per option), so its Choice and Score rates are far below Bool.
- **Score records are the longest** (HelpSteer dialogues), so every local model is slowest on them.
- One run on one machine; rerun to see the spread.